# RADS Verification Notebook 03: Verify Model

This notebook instantiates the `ClassificationModel` via `model_factory` (with ResNet-18), verifies the forward/backward passes, steps the optimizer and scheduler, and logs model info/learning rate to W&B.


In [1]:
import os
import sys
from pathlib import Path
import torch
import torch.nn as nn

# Ensure project root is in path
project_root = Path("../..").resolve()
sys.path.insert(0, str(project_root))

from training.configs.config import load_training_config
from training.models.model_factory import create_model
from training.losses.classification_loss import create_loss
from training.callbacks.lr_monitor import LearningRateMonitor
from training.utils.wandb_manager import TrainingWandbManager
from training.utils.device import get_device

config = load_training_config()
device = get_device()
print(f'Using device: {device}')


Using device: cpu


In [2]:
# Create model via factory
model = create_model(config).to(device)
print(f'Loaded backbone: {model.backbone_name}')
print(f'Number of classes: {model.num_classes}')
print(f'Feature dim: {model.get_feature_dim()}')


Loaded backbone: resnet18
Number of classes: 3
Feature dim: 512


In [3]:
# Forward pass verification
x = torch.randn(4, 3, 224, 224).to(device)
logits = model(x)
print('Logits shape:', logits.shape)
assert logits.shape == (4, config.num_classes), 'Incorrect logits shape!'
print('Forward pass successful!')


Logits shape: torch.Size([4, 3])
Forward pass successful!


In [4]:
# Loss & backward pass verification
loss_fn = create_loss(config, device=device)
targets = torch.tensor([0, 1, 2, 0]).to(device) # Mock targets
loss = loss_fn(logits, targets)
print(f'Loss value: {loss.item():.4f}')

# Check gradients flow
loss.backward()
grad_norms = [p.grad.norm().item() for p in model.parameters() if p.grad is not None]
print(f'Number of parameter tensors with gradients: {len(grad_norms)}')
assert len(grad_norms) > 0, 'No gradients computed! Check loss.backward()'
print('Backward pass successful (gradients computed)!')


Loss value: 1.3916
Number of parameter tensors with gradients: 62
Backward pass successful (gradients computed)!


In [5]:
# Optimizer + scheduler step verification
optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
lr_monitor = LearningRateMonitor()

# Read starting LR
start_lr = lr_monitor.step(optimizer, 0)
print(f'Starting learning rate: {start_lr}')

# Step optimizer & scheduler
optimizer.step()
scheduler.step()
next_lr = optimizer.param_groups[0]['lr']
print(f'LR after 1 step / epoch: {next_lr}')


Starting learning rate: 0.001
LR after 1 step / epoch: 0.001


In [6]:
# Log model architecture info and LR to W&B
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

wb_manager = TrainingWandbManager(config, 'verify_model')
with wb_manager:
    wb_manager.log_stats({
        'model/total_parameters': total_params,
        'model/trainable_parameters': trainable_params,
        'model/backbone': model.backbone_name,
    })
    wb_manager.log_learning_rate(start_lr, 0)
    print('Model stats logged to W&B successfully!')


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Amita nagar\_netrc.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Model stats logged to W&B successfully!


epoch,▁
lr/learning_rate,▁
model/total_parameters,▁
model/trainable_parameters,▁
epoch,0
lr/learning_rate,0.001
model/backbone,resnet18
model/total_parameters,11178051
model/trainable_parameters,11178051
